# Predict Seagrass at different times
* Designed to work year by year
* Pick a location and run

In [3]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [4]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:55470,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:55489,Total threads: 2
Dashboard: http://127.0.0.1:55490/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:55473,


# Standardise Council Boundaries

In [5]:
data_path = utils.get_data_path()
council_site_path = data_path / "site_polygons" / "council_sites" / "standardised"

council_sites = {"AshleyRiver_Estuary.zip/AshleyRiver_Estuary": "Ashley Mouth ECAN",
                 "Brookland_Lagoon.zip/Brookland Lagoon": "Brookland Lagoon ECAN",
                 "porirua_intertidal_extent_GW.zip": "Porirua Estuary GWRC",
                 "Nelson Haven from OSM.geojson": "Nelson Haven NCC",
                 "Manukau_estuary_polygon_MHWS.zip/Manukau_estuary_polygon_MHWS": "Manukau Estuary AC"}

for key, value in council_sites.items():
    if not (council_site_path / f"{value}.gpkg").exists():
        print(f"Save {value}")
        if "zip" in key:
            zip_path = council_site_path.parent / key
            boundary = geopandas.read_file(f"/vsizip/{zip_path.as_posix()}")
        else:
            boundary = geopandas.read_file(council_site_path.parent / key)
        boundary.to_crs(utils.CRS_NZTM).to_file(council_site_path / f"{value}.gpkg")
    else:
        print(f"Already created {value}")

Already created Ashley Mouth ECAN
Already created Brookland Lagoon ECAN
Already created Porirua Estuary GWRC
Already created Nelson Haven NCC
Already created Manukau Estuary AC


# Value to edit

In [7]:
prediction_sites = list(council_sites.values())
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [8]:
data_path = utils.get_data_path()
utils.create_data_folders()

In [10]:
prediction_sites[:4]

['Ashley Mouth ECAN',
 'Brookland Lagoon ECAN',
 'Porirua Estuary GWRC',
 'Nelson Haven NCC']

# Cells to run
* Get lowtide satellite images by year and site
* Predict Seagrass across satellite images

### Save satellite
The low-tide no cloud images across sites for a whole year

In [6]:
predict_max_cloud_covers = [1] # percentage
predict_tide_deltas = {1: {"hrs": 0, "mins": 30}}
filtered=True

In [12]:
all_baselines = pandas.read_csv(utils.get_data_path() / "council_band_quality_baseline.csv")

for tide_delta in predict_tide_deltas.values():
    for max_cloud_cover in predict_max_cloud_covers:
        print(f"Max cloud percentage {max_cloud_cover}, Low tide delta {tide_delta['hrs']}hrs and {tide_delta['mins']}mins")
        predictions_path, satellite_path = utils.get_prediction_path(
            sample_method="sampling_2", method_2_threshold=0.5,
            model_max_cloud_cover=max_cloud_cover,
            predict_max_cloud_cover=max_cloud_cover,
            predict_low_tide_delta_hrs=tide_delta["hrs"],
            predict_low_tide_delta_mins=tide_delta["mins"],
            use_threshold=False, satellite_only=True,
            filtered=filtered, use_council_paths=True)
        
        for prediction_site in prediction_sites[:4]:

            site_baseline = all_baselines.loc[all_baselines["site"] == prediction_site]
            
            for year in years:
                print(f"{prediction_site}: year {year}")
            
                satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
                site_polygon_file = council_site_path / f"{prediction_site}.gpkg"

                if (satellite_path / f"{prediction_site}_{years[-1]}_sentinel-2.nc").exists():
                    print(f"\tAlready run for {prediction_site} as {prediction_site}_{years[-1]}_sentinel-2.nc exists. "
                          "Delete if you want redownloaded.")
                    break
            
                if not satellite_file.exists():
                    print(f"\tSave satellite image of the site {prediction_site} around lowtide without cloud in the year {year}")
                    site_polygon = geopandas.read_file(site_polygon_file)
                    for i in range(3):
                        try:
                            satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                                geometry=site_polygon,
                                max_cloud_cover=max_cloud_cover,
                                year=year,
                                low_tide_delta_hrs=tide_delta["hrs"],
                                low_tide_delta_mins=tide_delta["mins"],
                                anomaly_filter_baseline=site_baseline,
                                n_mad=3,
                            )
                            break
                        except Exception:
                            if i == 2:
                                raise
                            print("Caught exception downloading images and try again.")
                    if len(satellite_data['time']) == 0:
                        print("Warning: No satellite data without cloud. Ignore")
                    else:
                        print("\t\tSave")
                        utils.write_netcdf_conventions_in_place(satellite_data)
                        satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                        satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                        utils.write_netcdf_conventions_in_place(satellite_data)
                        utils.save_netcdf(satellite_data, satellite_file)
                    utils.rgb_from_satellite(satellite_data_path=satellite_file)

Max cloud percentage 1, Low tide delta 0hrs and 30mins
Ashley Mouth ECAN: year 2016
	Already run for Ashley Mouth ECAN as Ashley Mouth ECAN_2026_sentinel-2.nc exists. Delete if you want redownloaded.
Brookland Lagoon ECAN: year 2016
	Already run for Brookland Lagoon ECAN as Brookland Lagoon ECAN_2026_sentinel-2.nc exists. Delete if you want redownloaded.
Porirua Estuary GWRC: year 2016
	Already run for Porirua Estuary GWRC as Porirua Estuary GWRC_2026_sentinel-2.nc exists. Delete if you want redownloaded.
Nelson Haven NCC: year 2016
	Already run for Nelson Haven NCC as Nelson Haven NCC_2026_sentinel-2.nc exists. Delete if you want redownloaded.


# Manakau separatly as no baseline and 10% cloud 
* As otherwise no suitable imagery

In [16]:
predict_max_cloud_covers = [10] # percentage
predict_tide_deltas = {1: {"hrs": 0, "mins": 30}}
filtered=False

In [17]:
for tide_delta in predict_tide_deltas.values():
    for max_cloud_cover in predict_max_cloud_covers:
        print(f"Max cloud percentage {max_cloud_cover}, Low tide delta {tide_delta['hrs']}hrs and {tide_delta['mins']}mins")
        predictions_path, satellite_path = utils.get_prediction_path(
            sample_method="sampling_2", method_2_threshold=0.5,
            model_max_cloud_cover=max_cloud_cover,
            predict_max_cloud_cover=max_cloud_cover,
            predict_low_tide_delta_hrs=tide_delta["hrs"],
            predict_low_tide_delta_mins=tide_delta["mins"],
            use_threshold=False, satellite_only=True,
            filtered=filtered, use_council_paths=True)
        
        for prediction_site in prediction_sites[-1:]:

            site_baseline = all_baselines.loc[all_baselines["site"] == prediction_site]
            
            for year in years:
                print(f"{prediction_site}: year {year}")
            
                satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
                site_polygon_file = council_site_path / f"{prediction_site}.gpkg"

                if (satellite_path / f"{prediction_site}_{years[-1]}_sentinel-2.nc").exists():
                    print(f"\tAlready run for {prediction_site} as {prediction_site}_{years[-1]}_sentinel-2.nc exists. "
                          "Delete if you want redownloaded.")
                    break
            
                if not satellite_file.exists():
                    print(f"\tSave satellite image of the site {prediction_site} around lowtide without cloud in the year {year}")
                    site_polygon = geopandas.read_file(site_polygon_file)
                    for i in range(3):
                        try:
                            satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                                geometry=site_polygon,
                                max_cloud_cover=max_cloud_cover,
                                year=year,
                                low_tide_delta_hrs=tide_delta["hrs"],
                                low_tide_delta_mins=tide_delta["mins"],
                            )
                            break
                        except Exception:
                            if i == 2:
                                raise
                            print("Caught exception downloading images and try again.")
                    if len(satellite_data['time']) == 0:
                        print("Warning: No satellite data without cloud. Ignore")
                    else:
                        print("\t\tSave")
                        utils.write_netcdf_conventions_in_place(satellite_data)
                        satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                        satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                        utils.write_netcdf_conventions_in_place(satellite_data)
                        utils.save_netcdf(satellite_data, satellite_file)
                    utils.rgb_from_satellite(satellite_data_path=satellite_file)

Max cloud percentage 10, Low tide delta 0hrs and 30mins
Manukau Estuary AC: year 2016
	Save satellite image of the site Manukau Estuary AC around lowtide without cloud in the year 2016
	Tide API-filtered tiles: 4 from 158 STAC items
	Tide API-filtered tiles: 1 from 158 STAC items
	Low cloud tiles: 0 from the low tide tiles of 1. Cloud percentages [39.8] Kept [False]
Satellite data path C:\Local\repos\seagrass-detection\data\council_predictions\tide_30mins_cloud_10\satellite_images\Manukau Estuary AC_2016_sentinel-2.nc does not exist.
Manukau Estuary AC: year 2017
	Save satellite image of the site Manukau Estuary AC around lowtide without cloud in the year 2017
	Tide API-filtered tiles: 8 from 187 STAC items
	Tide API-filtered tiles: 2 from 187 STAC items
	Low cloud tiles: 0 from the low tide tiles of 2. Cloud percentages [77.4 16.9] Kept [False False]
Satellite data path C:\Local\repos\seagrass-detection\data\council_predictions\tide_30mins_cloud_10\satellite_images\Manukau Estuary AC_